# Lab 4: Regression Analysis with Regularization Techniques

Name: Ashish Mahajan  
Course: 2026 Summer - Advanced Big Data and Data Mining (MSCS-634-M20)  
Dataset: Diabetes Dataset from sklearn

## Introduction
Regression predicts a continuous numeric target. In this lab, the Diabetes dataset contains health-related measurements used to predict a continuous disease progression value.

The models compared in this notebook are:
- Simple Linear Regression, which uses one feature.
- Multiple Linear Regression, which uses multiple features.
- Polynomial Regression, which creates nonlinear feature combinations.
- Ridge Regression, which uses L2 regularization to shrink coefficients.
- Lasso Regression, which uses L1 regularization and can shrink some coefficients to zero.

Overfitting happens when a model learns the training data too closely and performs poorly on unseen data. Underfitting happens when a model is too simple to capture useful patterns. Regularization helps control model complexity so the model can generalize better.


## Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

os.makedirs("screenshots", exist_ok=True)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## Helper Functions

In [ ]:
def evaluate_model(model_name, y_train, train_predictions, y_test, test_predictions):
    return {
        "Model": model_name,
        "Train MAE": mean_absolute_error(y_train, train_predictions),
        "Test MAE": mean_absolute_error(y_test, test_predictions),
        "Train MSE": mean_squared_error(y_train, train_predictions),
        "Test MSE": mean_squared_error(y_test, test_predictions),
        "Train RMSE": root_mean_squared_error(y_train, train_predictions),
        "Test RMSE": root_mean_squared_error(y_test, test_predictions),
        "Train R2": r2_score(y_train, train_predictions),
        "Test R2": r2_score(y_test, test_predictions),
    }


def plot_actual_vs_predicted(y_test, predictions, title, file_path):
    plt.figure(figsize=(7, 5))
    plt.scatter(y_test, predictions, alpha=0.75, color="#2f6f9f", edgecolor="white", linewidth=0.6)
    min_value = min(y_test.min(), predictions.min())
    max_value = max(y_test.max(), predictions.max())
    plt.plot([min_value, max_value], [min_value, max_value], color="#c44e52", linestyle="--", label="Ideal prediction")
    plt.title(title)
    plt.xlabel("Actual Disease Progression")
    plt.ylabel("Predicted Disease Progression")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(file_path, dpi=150)
    plt.show()


## Step 1: Load and Prepare the Dataset

The Diabetes dataset is loaded directly from sklearn. The dataset has 442 samples, 10 numeric features, and one continuous target value. Because the target is numeric, this is a regression problem.


In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data.copy()
y = diabetes.target.copy()
df = diabetes.frame.copy()

print("First five rows of the dataset:")
display(df.head())

print("Dataset shape:")
display(pd.DataFrame({"Rows": [df.shape[0]], "Columns": [df.shape[1]]}))

print("Feature names:")
display(pd.DataFrame({"Feature": diabetes.feature_names}))


In [ ]:
print("Dataset information:")
df.info()

print("Numeric summary for features and target:")
display(df.describe().round(4))

print("Target summary:")
display(y.describe().round(4).to_frame(name="target"))


In [ ]:
missing_values = df.isna().sum().reset_index()
missing_values.columns = ["Column", "Missing Values"]
missing_values


## Feature and Target Distribution Visualizations

These visualizations help us understand the target distribution, relationships between features and the target, and why `bmi` is a useful feature for the simple regression baseline.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(y, bins=25, color="#4c78a8", edgecolor="white")
plt.title("Distribution of Diabetes Disease Progression Target")
plt.xlabel("Disease Progression")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/04_target_distribution.png", dpi=150)
plt.show()


In [ ]:
feature_target_correlation = df.corr(numeric_only=True)["target"].drop("target").sort_values()

plt.figure(figsize=(9, 5))
colors = ["#c44e52" if value < 0 else "#4c78a8" for value in feature_target_correlation]
plt.barh(feature_target_correlation.index, feature_target_correlation.values, color=colors)
plt.title("Feature Correlation with Diabetes Progression Target")
plt.xlabel("Correlation with Target")
plt.ylabel("Feature")
plt.axvline(0, color="black", linewidth=0.8)
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/05_feature_target_correlation.png", dpi=150)
plt.show()

feature_target_correlation.round(4).to_frame(name="Correlation with Target")


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["bmi"], y, alpha=0.75, color="#59a14f", edgecolor="white", linewidth=0.5)
plt.title("BMI vs Diabetes Disease Progression")
plt.xlabel("BMI Feature")
plt.ylabel("Disease Progression")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/06_bmi_vs_target.png", dpi=150)
plt.show()


The target distribution helps us see the spread of disease progression values. The correlation chart helps identify useful predictors. The `bmi` feature is used for Simple Linear Regression because it has a clear relationship with the target and is easy to interpret visually.


## Train/Test Split

Training data is used to fit the model. Testing data is kept separate so we can evaluate how well the model performs on unseen data. The same train/test split is used across models for fair comparison.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

split_summary = pd.DataFrame({
    "Object": ["X_train", "X_test", "y_train", "y_test"],
    "Shape": [X_train.shape, X_test.shape, y_train.shape, y_test.shape]
})
split_summary


## Step 2: Simple Linear Regression

Simple Linear Regression uses one independent variable to predict the target. Here, `bmi` is used as a single-feature baseline. This helps show what a simple model can and cannot capture.


In [ ]:
results = []

simple_feature = "bmi"
X_simple = X[[simple_feature]]

X_simple_train, X_simple_test, y_simple_train, y_simple_test = train_test_split(
    X_simple,
    y,
    test_size=0.2,
    random_state=42
)

simple_model = LinearRegression()
simple_model.fit(X_simple_train, y_simple_train)

simple_train_predictions = simple_model.predict(X_simple_train)
simple_test_predictions = simple_model.predict(X_simple_test)

simple_result = evaluate_model(
    "Simple Linear Regression",
    y_simple_train,
    simple_train_predictions,
    y_simple_test,
    simple_test_predictions
)
results.append(simple_result)

simple_details = pd.DataFrame({
    "Metric": ["Feature", "Coefficient", "Intercept"],
    "Value": [simple_feature, simple_model.coef_[0], simple_model.intercept_]
})

display(simple_details)
display(pd.DataFrame([simple_result]).round(4))


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_simple_train[simple_feature], y_simple_train, alpha=0.55, color="#4c78a8", label="Training data")
plt.scatter(X_simple_test[simple_feature], y_simple_test, alpha=0.75, color="#f58518", label="Testing data")

x_line = np.linspace(X_simple[simple_feature].min(), X_simple[simple_feature].max(), 100).reshape(-1, 1)
y_line = simple_model.predict(pd.DataFrame(x_line, columns=[simple_feature]))
plt.plot(x_line, y_line, color="#c44e52", linewidth=2, label="Regression line")

plt.title("Simple Linear Regression: BMI vs Disease Progression")
plt.xlabel("BMI Feature")
plt.ylabel("Disease Progression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/08_simple_linear_regression_line.png", dpi=150)
plt.show()


In [ ]:
plot_actual_vs_predicted(
    y_simple_test,
    simple_test_predictions,
    "Simple Linear Regression: Actual vs Predicted",
    "screenshots/09_simple_actual_vs_predicted.png"
)


The regression line shows the general trend between `bmi` and disease progression. Because this model uses only one feature, it is easy to interpret, but it may underfit because it ignores the other available health measurements.


## Step 3: Multiple Linear Regression

Multiple Linear Regression uses all 10 features to predict disease progression. A pipeline with `StandardScaler` keeps preprocessing and modeling together, which makes the workflow easier to repeat and compare.


In [ ]:
multiple_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

multiple_model.fit(X_train, y_train)
multiple_train_predictions = multiple_model.predict(X_train)
multiple_test_predictions = multiple_model.predict(X_test)

multiple_result = evaluate_model(
    "Multiple Linear Regression",
    y_train,
    multiple_train_predictions,
    y_test,
    multiple_test_predictions
)
results.append(multiple_result)

multiple_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": multiple_model.named_steps["model"].coef_
})
multiple_coefficients["Absolute Coefficient"] = multiple_coefficients["Coefficient"].abs()
multiple_coefficients = multiple_coefficients.sort_values("Absolute Coefficient", ascending=False)

display(pd.DataFrame([multiple_result]).round(4))
display(multiple_coefficients.round(4))


In [ ]:
plot_actual_vs_predicted(
    y_test,
    multiple_test_predictions,
    "Multiple Linear Regression: Actual vs Predicted",
    "screenshots/10_multiple_actual_vs_predicted.png"
)


In [ ]:
plt.figure(figsize=(9, 5))
coefficient_plot_data = multiple_coefficients.sort_values("Coefficient")
colors = ["#c44e52" if value < 0 else "#4c78a8" for value in coefficient_plot_data["Coefficient"]]
plt.barh(coefficient_plot_data["Feature"], coefficient_plot_data["Coefficient"], color=colors)
plt.title("Multiple Linear Regression Coefficients")
plt.xlabel("Standardized Coefficient")
plt.ylabel("Feature")
plt.axvline(0, color="black", linewidth=0.8)
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/11_multiple_coefficients.png", dpi=150)
plt.show()


Multiple Regression uses more information than Simple Linear Regression, so it can capture more relationships in the dataset. The coefficient chart helps show which features have larger positive or negative effects after scaling.


## Step 4: Polynomial Regression

Polynomial Regression creates nonlinear feature transformations. In this section, the model still uses only `bmi` so the curved relationships can be visualized clearly. Degrees 2, 3, and 4 are compared to show how increasing complexity affects performance.


In [ ]:
degrees = [2, 3, 4]
polynomial_results = []
polynomial_models = {}

for degree in degrees:
    polynomial_model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])
    
    polynomial_model.fit(X_simple_train, y_simple_train)
    polynomial_train_predictions = polynomial_model.predict(X_simple_train)
    polynomial_test_predictions = polynomial_model.predict(X_simple_test)
    
    polynomial_result = evaluate_model(
        f"Polynomial Regression degree {degree}",
        y_simple_train,
        polynomial_train_predictions,
        y_simple_test,
        polynomial_test_predictions
    )
    polynomial_result["Degree"] = degree
    polynomial_results.append(polynomial_result)
    results.append({key: value for key, value in polynomial_result.items() if key != "Degree"})
    polynomial_models[degree] = polynomial_model

polynomial_results_df = pd.DataFrame(polynomial_results).round(4)
polynomial_results_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_simple_train[simple_feature], y_simple_train, alpha=0.45, color="#9ecae1", label="Training data")
plt.scatter(X_simple_test[simple_feature], y_simple_test, alpha=0.65, color="#fdae6b", label="Testing data")

x_grid = np.linspace(X_simple[simple_feature].min(), X_simple[simple_feature].max(), 200)
x_grid_df = pd.DataFrame({simple_feature: x_grid})
curve_colors = {2: "#4c78a8", 3: "#59a14f", 4: "#c44e52"}

for degree in degrees:
    y_curve = polynomial_models[degree].predict(x_grid_df)
    plt.plot(x_grid, y_curve, linewidth=2, color=curve_colors[degree], label=f"Degree {degree}")

plt.title("Polynomial Regression Curves Using BMI")
plt.xlabel("BMI Feature")
plt.ylabel("Disease Progression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/12_polynomial_regression_curves.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(polynomial_results_df["Degree"], polynomial_results_df["Train R2"], marker="o", label="Train R2", color="#4c78a8")
plt.plot(polynomial_results_df["Degree"], polynomial_results_df["Test R2"], marker="o", label="Test R2", color="#c44e52")
plt.title("Polynomial Degree vs Train and Test R-squared")
plt.xlabel("Polynomial Degree")
plt.ylabel("R-squared")
plt.xticks(degrees)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/13_polynomial_degree_r2.png", dpi=150)
plt.show()


Polynomial features allow nonlinear curves. If training R-squared improves while testing R-squared does not improve, that can suggest overfitting. The actual Train R2 and Test R2 values above should be used when writing the final interpretation.


## Step 5: Regularization with Ridge and Lasso Regression

Ridge Regression uses L2 regularization and shrinks coefficients without usually forcing them to zero. Lasso Regression uses L1 regularization and can set some coefficients to zero. The alpha value controls regularization strength, and larger alpha values usually increase coefficient shrinkage.


In [ ]:
alpha_values = [0.01, 0.1, 1, 10, 100]

ridge_results = []
ridge_models = {}

for alpha in alpha_values:
    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    
    ridge_model.fit(X_train, y_train)
    ridge_train_predictions = ridge_model.predict(X_train)
    ridge_test_predictions = ridge_model.predict(X_test)
    
    ridge_result = evaluate_model(
        f"Ridge Regression alpha {alpha}",
        y_train,
        ridge_train_predictions,
        y_test,
        ridge_test_predictions
    )
    ridge_coefficients = ridge_model.named_steps["model"].coef_
    ridge_result["Alpha"] = alpha
    ridge_result["Nonzero Coefficients"] = np.sum(np.abs(ridge_coefficients) > 1e-6)
    ridge_results.append(ridge_result)
    ridge_models[alpha] = ridge_model

ridge_results_df = pd.DataFrame(ridge_results).sort_values("Test RMSE").reset_index(drop=True)
ridge_results_df.round(4)


In [ ]:
lasso_results = []
lasso_models = {}

for alpha in alpha_values:
    lasso_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=10000, random_state=42))
    ])
    
    lasso_model.fit(X_train, y_train)
    lasso_train_predictions = lasso_model.predict(X_train)
    lasso_test_predictions = lasso_model.predict(X_test)
    
    lasso_result = evaluate_model(
        f"Lasso Regression alpha {alpha}",
        y_train,
        lasso_train_predictions,
        y_test,
        lasso_test_predictions
    )
    lasso_coefficients = lasso_model.named_steps["model"].coef_
    lasso_result["Alpha"] = alpha
    lasso_result["Nonzero Coefficients"] = np.sum(np.abs(lasso_coefficients) > 1e-6)
    lasso_results.append(lasso_result)
    lasso_models[alpha] = lasso_model

lasso_results_df = pd.DataFrame(lasso_results).sort_values("Test RMSE").reset_index(drop=True)
lasso_results_df.round(4)


In [ ]:
best_ridge = ridge_results_df.iloc[0]
best_lasso = lasso_results_df.iloc[0]
best_ridge_alpha = best_ridge["Alpha"]
best_lasso_alpha = best_lasso["Alpha"]

best_ridge_model = ridge_models[best_ridge_alpha]
best_lasso_model = lasso_models[best_lasso_alpha]

best_ridge_train_predictions = best_ridge_model.predict(X_train)
best_ridge_test_predictions = best_ridge_model.predict(X_test)
best_lasso_train_predictions = best_lasso_model.predict(X_train)
best_lasso_test_predictions = best_lasso_model.predict(X_test)

best_ridge_result = evaluate_model(
    "Best Ridge Regression",
    y_train,
    best_ridge_train_predictions,
    y_test,
    best_ridge_test_predictions
)
best_ridge_result["Alpha"] = best_ridge_alpha

best_lasso_result = evaluate_model(
    "Best Lasso Regression",
    y_train,
    best_lasso_train_predictions,
    y_test,
    best_lasso_test_predictions
)
best_lasso_result["Alpha"] = best_lasso_alpha

results.append(best_ridge_result)
results.append(best_lasso_result)

best_regularized_summary = pd.DataFrame([best_ridge_result, best_lasso_result]).round(4)
best_regularized_summary


In [ ]:
plt.figure(figsize=(8, 5))
ridge_alpha_plot = pd.DataFrame(ridge_results).sort_values("Alpha")
plt.plot(ridge_alpha_plot["Alpha"], ridge_alpha_plot["Test RMSE"], marker="o", color="#4c78a8")
plt.xscale("log")
plt.title("Ridge Alpha vs Test RMSE")
plt.xlabel("Alpha")
plt.ylabel("Test RMSE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/14_ridge_alpha_rmse.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
lasso_alpha_plot = pd.DataFrame(lasso_results).sort_values("Alpha")
plt.plot(lasso_alpha_plot["Alpha"], lasso_alpha_plot["Test RMSE"], marker="o", color="#c44e52")
plt.xscale("log")
plt.title("Lasso Alpha vs Test RMSE")
plt.xlabel("Alpha")
plt.ylabel("Test RMSE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/15_lasso_alpha_rmse.png", dpi=150)
plt.show()


In [ ]:
ridge_lasso_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Best Ridge Coefficient": best_ridge_model.named_steps["model"].coef_,
    "Best Lasso Coefficient": best_lasso_model.named_steps["model"].coef_
})

x_positions = np.arange(len(ridge_lasso_coefficients))
bar_width = 0.38

plt.figure(figsize=(10, 5))
plt.bar(x_positions - bar_width / 2, ridge_lasso_coefficients["Best Ridge Coefficient"], width=bar_width, label=f"Ridge alpha {best_ridge_alpha}", color="#4c78a8")
plt.bar(x_positions + bar_width / 2, ridge_lasso_coefficients["Best Lasso Coefficient"], width=bar_width, label=f"Lasso alpha {best_lasso_alpha}", color="#c44e52")
plt.title("Best Ridge and Lasso Coefficient Comparison")
plt.xlabel("Feature")
plt.ylabel("Standardized Coefficient")
plt.xticks(x_positions, ridge_lasso_coefficients["Feature"], rotation=45, ha="right")
plt.axhline(0, color="black", linewidth=0.8)
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/16_ridge_lasso_coefficients.png", dpi=150)
plt.show()

ridge_lasso_coefficients.round(4)


In [ ]:
plot_actual_vs_predicted(
    y_test,
    best_ridge_test_predictions,
    "Best Ridge Regression: Actual vs Predicted",
    "screenshots/17_ridge_actual_vs_predicted.png"
)


In [ ]:
plot_actual_vs_predicted(
    y_test,
    best_lasso_test_predictions,
    "Best Lasso Regression: Actual vs Predicted",
    "screenshots/18_lasso_actual_vs_predicted.png"
)


Ridge and Lasso show how regularization changes model performance and coefficient behavior. Ridge usually keeps all coefficients but makes them smaller. Lasso can reduce some coefficients to zero, which can also support feature selection.


## Cross-Validation

Cross-validation evaluates whether model performance is stable across different splits. This supports the final recommendation because it gives another view beyond one train/test split.


In [ ]:
cv_models = {
    "Multiple Linear Regression": multiple_model,
    f"Best Ridge Regression alpha {best_ridge_alpha}": best_ridge_model,
    f"Best Lasso Regression alpha {best_lasso_alpha}": best_lasso_model,
}

cv_results = []
for model_name, model in cv_models.items():
    cv_scores = cross_val_score(
        model,
        X,
        y,
        scoring="neg_root_mean_squared_error",
        cv=5
    )
    cv_rmse_scores = -cv_scores
    cv_results.append({
        "Model": model_name,
        "Mean CV RMSE": cv_rmse_scores.mean(),
        "CV RMSE Std": cv_rmse_scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("Mean CV RMSE").reset_index(drop=True)
cv_results_df.round(4)


## Step 6: Model Comparison and Analysis

The final comparison table brings all model results together. Lower Test RMSE is better, and higher Test R2 is better. Train R2 and Test R2 are compared to check whether a model may be overfitting.


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[[
    "Model",
    "Train MAE",
    "Test MAE",
    "Train MSE",
    "Test MSE",
    "Train RMSE",
    "Test RMSE",
    "Train R2",
    "Test R2"
] + (["Alpha"] if "Alpha" in results_df.columns else [])]

results_df_sorted = results_df.sort_values("Test RMSE").reset_index(drop=True)
results_df_sorted.round(4)


In [ ]:
plt.figure(figsize=(10, 5))
rmse_plot_data = results_df_sorted.sort_values("Test RMSE", ascending=False)
plt.barh(rmse_plot_data["Model"], rmse_plot_data["Test RMSE"], color="#4c78a8")
plt.title("Model Comparison by Test RMSE")
plt.xlabel("Test RMSE")
plt.ylabel("Model")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/20_model_comparison_rmse.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
r2_plot_data = results_df_sorted.sort_values("Test R2")
plt.barh(r2_plot_data["Model"], r2_plot_data["Test R2"], color="#59a14f")
plt.title("Model Comparison by Test R-squared")
plt.xlabel("Test R-squared")
plt.ylabel("Model")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/21_model_comparison_r2.png", dpi=150)
plt.show()


In [ ]:
r2_comparison = results_df_sorted[["Model", "Train R2", "Test R2"]].set_index("Model")

plt.figure(figsize=(11, 5))
x_positions = np.arange(len(r2_comparison.index))
bar_width = 0.38
plt.bar(x_positions - bar_width / 2, r2_comparison["Train R2"], width=bar_width, label="Train R2", color="#4c78a8")
plt.bar(x_positions + bar_width / 2, r2_comparison["Test R2"], width=bar_width, label="Test R2", color="#c44e52")
plt.title("Train vs Test R-squared by Model")
plt.xlabel("Model")
plt.ylabel("R-squared")
plt.xticks(x_positions, r2_comparison.index, rotation=45, ha="right")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("screenshots/22_train_vs_test_r2.png", dpi=150)
plt.show()


In [ ]:
best_rmse_row = results_df_sorted.iloc[0]
best_r2_row = results_df.sort_values("Test R2", ascending=False).iloc[0]

final_summary = pd.DataFrame({
    "Question": [
        "Best model by Test RMSE",
        "Best model by Test R2",
        "Best Ridge alpha",
        "Best Lasso alpha"
    ],
    "Answer": [
        f"{best_rmse_row['Model']} with Test RMSE {best_rmse_row['Test RMSE']:.4f}",
        f"{best_r2_row['Model']} with Test R2 {best_r2_row['Test R2']:.4f}",
        best_ridge_alpha,
        best_lasso_alpha
    ]
})
final_summary
